#LOAD_Preparation_Data

*Propósito:* Construir la tabla maestra de features (portafolio + variables) aplicando lógica de negocio: Dueños, materialidad, variables AG, antiguedad SUNAT,etc. Escritura particionada por codmes.

In [0]:
%run ../config/config

In [0]:
import sys
import time
import os
import numpy as np
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType, StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import udf, array

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.month_delta import month_delta

In [0]:
# Logger load preparation data

logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO",
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '03_load_preparation_data'
    }
)

In [0]:
def add_codmes_spark(col_name, months):
    return F.expr(f"""    
        cast(
            date_format(
                add_months(
                    to_date(cast({col_name} as string), 'yyyyMM'),
                    {months}
                ),
                'yyyymm'
            ) as int
        )
    """)

def operacionesMaxBetweenCols(columnas):
    """UDF para calcular el máximo entre columnas manejando nulos (versión reference)."""
    resultado = 0.0
    valorInicial = float(-99999999999999.0)
    mayor = float(0.0)
    numero = float(0.0)
    for column in columnas:
        if column is not None and str(column) != "" and str(column).upper() != "NULL":
            numero = float(column)
        else:
            numero = -99999999999999.0
        if numero >= valorInicial:
            mayor = float(numero)
            valorInicial = mayor
        elif numero < valorInicial:
            mayor = float(valorInicial)
    if mayor == -99999999999999.0:
        resultado = None
    else:
        resultado = mayor
    return resultado

operacionesMaxBetweenCols_udf = udf(operacionesMaxBetweenCols, DoubleType())

In [0]:
RUN_HISTORICAL = FIRST_RUN
if RUN_HISTORICAL and MESES_HISTORICOS_CALIDAD > 0:
    meses_historicos_lista = []
    mes_tmp = MES_VTA
    for i in range(MESES_HISTORICOS_CALIDAD):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)
    meses_historicos_lista.reverse()
    meses_a_procesar = meses_historicos_lista + [MES_VTA]
    logger.info(f"MODO HISTÓRICO: {len(meses_a_procesar)} meses: {meses_a_procesar}")
else:
    meses_a_procesar = [MES_VTA]
    logger.info(f"Procesando únicamente el mes actual: {MES_VTA}")

spark = SparkSession.builder.getOrCreate()
resultados = []

for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("*" * 80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("*" * 80)

    try:
        #logger.log_stage_start('load_preparation_data', MES_VTA=mes_actual_proceso, #environment=ENV)
        start_time = time.time()

        # ================================================================
        # 1. CONFIGURACIÓN DEL PIPELINE
        # ================================================================
        # Reemplazar con la configuración específica de tu pipeline
        # Ejemplo:
        # from utils.data_preparation.config import PipelineConfig
        # config = PipelineConfig(codmes_ventas_actual_proceso, ...)
        # config.summary()

        # ================================================================
        # 2. EJECUCIÓN DE PREPARACIÓN DE DATOS (CUSTOMIZABLE POR MODELO)
        # ================================================================
        # Aquí debes importar y ejecutar tus clases de preparación de datos.
        # Ejemplo:
        # from utils.data_preparation.my_feature_extractor import MyFeatureExtractor
        # extractor = MyFeatureExtractor(spark, config)
        # extractor.run()

        logger.info(f"Ejecutando preparación de datos para codmes={mes_actual_proceso}")

        # IMPORTANTE: Al finalizar tu proceso, debes asegurar que los resultados
        # se escriban en la tabla definida en TABLE_MASTER u otra tabla de salida esperada.
        # ================================================================

        # Final count for logging (Asegúrate de que TABLE_MASTER haya sido creada)
        # Final count for logging
        count_final = spark.table(TABLE_MASTER).count()
        logger.info(f"Registros finales: {count_final};")

        total_duration = time.time() - start_time
        logger.log_stage_end('load_preparation_data', duration=total_duration, records_extracted=count_final)
        resultados.append({'mes': mes_actual_proceso, 'registros': count_final, 'exitoso': True, 'duracion': total_duration})

except Exception as e:
    logger.log_exception(
        operation='load_preparation_data',
        exception=e,
        should_raise=False,
        MES_VTA=mes_actual_proceso
        )
    resultados.append({
        'mes': mes_actual_proceso,
        'registros': 0,
        'exitoso': False,
        'error': str(e)
        })
    continue


In [0]:
logger.info("In" + "=" * 80)
logger.info("RESUMEN FINAL")
logger.info("=" * 80)
exitosos = sum(1 for r in resultados if r['exitoso'])
logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")
for r in resultados:
    if r['exitoso']:
        logger.info(f" ✅ {r['mes' ]}: {r['registros' ]:,} registros en {r['duracion'] :.2f}s")
    else:
        logger.error(f" ❌ {r['mes']}: {r.get('error', '?')}")
        
if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger._flush_all_handlers()
    logger.close()

In [0]:
logger.info("\n" + "=" * 80)
logger.info("RESUMEN FINAL")
logger.info("=" * 80)
exitosos = sum(1 for r in resultados if r['exitoso'])
logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")
for r in resultados:
    if r['exitoso']:
        logger.info(f" ✓ {r['mes']}: {r['registros']:.} registros en {r['duracion']:.2f}s")
    else:
        logger.error(f" ✗ {r['mes']}: {r.get('error', '?')}")
if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger._flush_all_handlers()
    logger.close()

In [0]:
# Databricks notebook source
# ============================================================================
# 03_load_preparation_data.ipynb
# Propósito: Construir la tabla maestra de features (portafolio + variables)
#            aplicando lógica de negocio: dueños, materialidad, variables AG,
#            antigüedad SUNAT, módulo activo, capping, etc.
#            
#            Versión refactorizada: toda la lógica está en clases dentro de
#            utils/data_preparation/.
# ============================================================================

# COMMAND ----------

%run ../config/config

# COMMAND ----------

# Imports
import sys
import time
import numpy as np
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType, StringType

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.month_delta import month_delta

# Importar todas las clases necesarias
from utils.data_preparation.universe_builder import UniverseBuilder
from utils.data_preparation.relationship_builder import RelationshipBuilder
from utils.data_preparation.demographic_builder import DemographicBuilder
from utils.data_preparation.carretera_builder import CarreteraBuilder
from utils.data_preparation.owner_variables_builder import OwnerVariablesBuilder
from utils.data_preparation.aggregated_variables_builder import AggregatedVariablesBuilder
from utils.data_preparation.sunat_seniority_builder import SunatSeniorityBuilder
from utils.data_preparation.additional_variables_builder import AdditionalVariablesBuilder
from utils.data_preparation.modulo_activo_builder import ModuloActivoBuilder
from utils.data_preparation.data_cleaner import DataCleaner
from utils.data_preparation.feature_selector import FeatureSelector
from utils.data_preparation.table_writer import TableWriter

# COMMAND ----------

# Logger
logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '03_load_preparation_data'
    }
)

# COMMAND ----------

# ================================================================
# PARÁMETROS DE EJECUCIÓN (histórico o solo actual)
# ================================================================
if FIRST_RUN and MESES_HISTORICOS_CALIDAD > 0:
    meses_historicos_lista = []
    mes_tmp = MES_VTA
    for i in range(MESES_HISTORICOS_CALIDAD):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)
    meses_historicos_lista.reverse()
    meses_a_procesar = meses_historicos_lista + [MES_VTA]
    logger.info(f"MODO HISTÓRICO: {len(meses_a_procesar)} meses: {meses_a_procesar}")
else:
    meses_a_procesar = [MES_VTA]
    logger.info(f"Procesando únicamente el mes actual: {MES_VTA}")

spark = SparkSession.builder.getOrCreate()
resultados = []

# Nombre de la tabla de salida (incluye sufijo _v2 si está configurado)
OUTPUT_FEATURES_TABLE = TABLE_MASTER  # TABLE_MASTER ya incluye TABLE_VERSION_SUFFIX desde config
logger.info(f"Tabla de salida: {OUTPUT_FEATURES_TABLE}")

# ================================================================
# INSTANCIAR TODOS LOS BUILDERS
# ================================================================
universe_builder = UniverseBuilder(spark, logger)
relationship_builder = RelationshipBuilder(spark, logger)
demographic_builder = DemographicBuilder(spark, logger)
carretera_builder = CarreteraBuilder(spark, logger)
owner_vars_builder = OwnerVariablesBuilder(spark, logger)
ag_vars_builder = AggregatedVariablesBuilder(spark, logger)
sunat_builder = SunatSeniorityBuilder(spark, logger)
additional_builder = AdditionalVariablesBuilder(spark, logger)
modulo_activo_builder = ModuloActivoBuilder(spark, logger)
data_cleaner = DataCleaner(spark, logger, caps_config={})  # Los umbrales de capping se definen dentro del cleaner
feature_selector = FeatureSelector(logger)
table_writer = TableWriter(logger)

# ================================================================
# BUCLE PRINCIPAL POR CADA MES
# ================================================================
for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("=" * 80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("=" * 80)

    try:
        logger.log_stage_start('load_preparation_data', mes_vta=mes_actual_proceso, environment=ENV)
        start_time = time.time()

        # Mes de datos = 1 mes antes del mes proceso (desfase de 1 mes)
        mes_data = int(month_delta(str(mes_actual_proceso), -1))
        logger.info(f"Mes proceso: {mes_actual_proceso} | Mes data (desfase -1): {mes_data}")

        # ====================================================
        # 1. UNIVERSO
        # ====================================================
        universo = universe_builder.build(SOURCE_TABLE, mes_actual_proceso)

        # ====================================================
        # 2. RELACIONES (DUEÑOS)
        # ====================================================
        relacion = relationship_builder.build(mes_data)

        # ====================================================
        # 3. VARIABLES DEMOGRÁFICAS
        # ====================================================
        demographic = demographic_builder.build(mes_data)

        # ====================================================
        # 4. CARRETERA (variables base con materialidad)
        # ====================================================
        carretera = carretera_builder.build(mes_data)

        # ====================================================
        # 5. VARIABLES DEL DUEÑO (_RL)
        # ====================================================
        owner_vars = owner_vars_builder.build(relacion, carretera)

        # ====================================================
        # 6. VARIABLES AGREGADAS (_AG)
        # ====================================================
        base = ag_vars_builder.build(universo, demographic, owner_vars, carretera)

        # ====================================================
        # 7. ANTIGÜEDAD SUNAT
        # ====================================================
        sunat = sunat_builder.build(mes_data)
        base = base.join(sunat, ["codclaveunicocli", "codmes"], "left_outer")

        # ====================================================
        # 8. TABLAS ADICIONALES (VIDEA, matrices, movimientos, etc.)
        # ====================================================
        base = additional_builder.join_all(base, mes_actual_proceso)

        # ====================================================
        # 9. MÓDULO ACTIVO
        # ====================================================
        modulo_activo = modulo_activo_builder.build(universo, mes_actual_proceso, mes_data)
        base = base.join(modulo_activo, ["codmes", "codclaveunicocli"], "left_outer")

        # ====================================================
        # 10. LIMPIEZA (dummy, actividad económica, capping)
        # ====================================================
        dummy_list = [
            1111111111, -1111111111, 2222222222, -2222222222, 3333333333, -3333333333,
            4444444444, 5555555555, 6666666666, 7777777777, -99, -999, 44444.4444, 55555.5555,
            66666.6666, 77777.7777, 11111.1111, -11111.1111, 22222.2222, -22222.2222,
            33333.3333, -33333.3333
        ]
        base_clean = data_cleaner.run(base, dummy_list)

        # ====================================================
        # 11. SELECCIÓN DE FEATURES (45 columnas finales)
        # ====================================================
        features_df = feature_selector.select(base_clean, FEATURE_NAMES)

        # ====================================================
        # 12. CONVERSIÓN DE CODMES A ENTERO (por si acaso)
        # ====================================================
        if 'codmes' in features_df.columns:
            sample = features_df.select(F.col("codmes")).first()[0]
            if isinstance(sample, str) and '-' in sample:
                features_df = features_df.withColumn(
                    "codmes",
                    (F.year(F.to_date(F.col("codmes"))) * 100 + F.month(F.to_date(F.col("codmes")))).cast("int")
                )
            else:
                features_df = features_df.withColumn("codmes", F.col("codmes").cast("int"))
        else:
            raise ValueError("La columna 'codmes' no existe en features_df")

        # ====================================================
        # 13. ESCRITURA EN TABLA DELTA
        # ====================================================
        if idx_mes == 1:
            write_mode = "overwrite"
            logger.info(f"Primer mes ({mes_actual_proceso}) - Modo: OVERWRITE")
        else:
            write_mode = "append"
            logger.info(f"Mes {mes_actual_proceso} - Modo: APPEND")

        table_writer.write(features_df, OUTPUT_FEATURES_TABLE, write_mode)

        count_final = features_df.count()
        logger.info(f"Registros escritos para mes {mes_actual_proceso}: {count_final:,}")
        logger.info(f"Tabla actualizada: {OUTPUT_FEATURES_TABLE}")

        total_duration = time.time() - start_time
        logger.log_stage_end('load_preparation_data', duration=total_duration, records_extracted=count_final)
        resultados.append({
            'mes': mes_actual_proceso,
            'registros': count_final,
            'exitoso': True,
            'duracion': total_duration
        })

    except Exception as e:
        logger.log_exception(
            operation='load_preparation_data',
            exception=e,
            should_raise=False,
            mes_vta=mes_actual_proceso
        )
        resultados.append({
            'mes': mes_actual_proceso,
            'registros': 0,
            'exitoso': False,
            'error': str(e)
        })
        continue

# ================================================================
# VISUALIZACIÓN DE LA TABLA GENERADA (FUERA DEL BUCLE - UNA SOLA VEZ)
# ================================================================
logger.info("")
logger.info("=" * 80)
logger.info("VISUALIZACIÓN DE LA TABLA GENERADA")
logger.info("=" * 80)

try:
    # 1. Cargar la tabla final
    df_verificacion = spark.table(OUTPUT_FEATURES_TABLE)
    
    # 2. Número de columnas
    num_cols = len(df_verificacion.columns)
    logger.info(f"📊 Número total de columnas en la tabla: {num_cols}")
    
    # 3. Lista de nombres de columnas
    logger.info("📋 Nombres de todas las columnas:")
    logger.info(", ".join(df_verificacion.columns))
    
    # 4. Mostrar las primeras 5 filas (en formato tabla)
    logger.info("🔍 Muestra de los primeros 5 registros:")
    display(df_verificacion.limit(5))
    
    # 5. Verificar que las 45 features están presentes
    if 'FEATURE_NAMES' in globals() and FEATURE_NAMES:
        columnas_tabla = set(df_verificacion.columns)
        features_set = set(FEATURE_NAMES)
        faltantes = features_set - columnas_tabla
        if faltantes:
            logger.warning(f"⚠️ Faltan {len(faltantes)} variables esperadas: {list(faltantes)}")
        else:
            logger.info(f"✅ Todas las {len(FEATURE_NAMES)} variables de FEATURE_NAMES están presentes")
    
    # 6. Conteo de registros por partición (codmes)
    logger.info("📁 Distribución de registros por partición (codmes):")
    display(df_verificacion.groupBy("codmes").count().orderBy("codmes"))
    
    # 7. Guardar métricas en task values
    dbutils.jobs.taskValues.set(key="features_table_columns", value=num_cols)
    dbutils.jobs.taskValues.set(key="features_table_name", value=OUTPUT_FEATURES_TABLE)
    dbutils.jobs.taskValues.set(key="features_total_records", value=df_verificacion.count())
    
    logger.info("✅ Visualización completada exitosamente")
    
except Exception as e:
    logger.error(f"❌ Error al visualizar la tabla: {e}")

# ================================================================
# RESUMEN FINAL
# ================================================================
logger.info("")
logger.info("=" * 80)
logger.info("RESUMEN FINAL")
logger.info("=" * 80)
exitosos = sum(1 for r in resultados if r['exitoso'])
logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")
for r in resultados:
    if r['exitoso']:
        logger.info(f" ✅ {r['mes']}: {r['registros']:,} registros en {r['duracion']:.2f}s")
    else:
        logger.error(f" ❌ {r['mes']}: {r.get('error', '?')}")

if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger._flush_all_handlers()
    logger.close()